#Setup

In [ ]:
# Run from the repository root.

In [ ]:
from google.colab import userdata
hf_token = userdata.get('hugging_face')
openai_token = userdata.get('openai')

In [ ]:
hf_token = ""
openai_token = ""

In [ ]:
#import dataset
import pandas as pd

df = pd.read_csv('data/raw/CS263_dataset.csv')
df.head()

#DeBERTa 2 Class + Threshold
We chose DeBERTa as one of our base models because its a model that is optimized for Natural Language Inference, trained to handle entailment-style reasoning.

**Design**

Have DeBERTa predict between "acceptable" and "not acceptable", if the confidence is below a threshold, return "context-dependent" instead.

In [ ]:
import re
import torch
import pandas as pd
from transformers import pipeline

# 1. Load dataset
df = pd.read_csv('data/raw/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load DeBERTa MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="microsoft/deberta-large-mnli",
    token=hf_token
)

candidate_labels = ["acceptable behavior", "unacceptable behavior"]

def predict_acceptability(situation, threshold=0.65):
    result = classifier(
        situation,
        candidate_labels=[
            "acceptable behavior",
            "not acceptable behavior"
        ],
        hypothesis_template="This behavior is {}."
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < threshold:
        return "context-dependent"

    if top_label == "acceptable behavior":
        return "acceptable"
    else:
        return "not acceptable"

# 5. Predict
predictions = []

for _, row in df.iterrows():

    with_context = f"{row['cultural_context']} {row['situation']}"
    pred_label = predict_acceptability(with_context)

    # pred_label = predict_acceptability(row["situation"])

    predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

# 6. Create result DataFrame
results_df = pd.DataFrame(predictions)

results_df.head()
results_df.to_csv("deberta_predictions_with_context.csv", index=False)

In [ ]:
!pip install openai

#ChatGPT

In [ ]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt(situation, cultural_context, culture):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt(row["situation"], row["cultural_context"], row["culture"])
    # pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()
gpt_results_df.to_csv("gpt_predictions_with_context.csv", index=False)

#Accuracy Analysis

In [ ]:
from sklearn.metrics import accuracy_score

# DeBERTa
deberta_acc = accuracy_score(
    results_df["gold_label"],
    results_df["prediction_label"]
)

# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"DeBERTa Accuracy: {deberta_acc:.4f}")
print(f"GPT Accuracy: {gpt_acc:.4f}")

In [ ]:
with open("accuracy_summary_with_context.txt", "w") as f:
    f.write(f"DeBERTa Accuracy: {deberta_acc:.4f}\n")
    f.write(f"GPT Accuracy: {gpt_acc:.4f}\n")

#Evaluation

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

labels = ["acceptable", "unacceptable", "context-dependent"]

def evaluate_model(df, model_name, save_path):
    y_true = df["gold_label"]
    y_pred = df["prediction_label"]

    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Classification report
    report = classification_report(y_true, y_pred, labels=labels)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Print results
    print(f"\n===== {model_name} =====")
    print(f"Accuracy: {acc:.4f}\n")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    # Save to file
    with open(save_path, "w") as f:
        f.write(f"===== {model_name} =====\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm))

# Run evaluations
evaluate_model(results_df, "DeBERTa", "deberta_eval_with_context.txt")
evaluate_model(gpt_results_df, "GPT", "gpt_eval_with_context.txt")

Aggregate Result

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/CS263_dataset.csv")

deberta_df = pd.read_csv("experiments/gpt_baselines/results/context_appended/deberta_predictions_with_context.csv")
gpt_df = pd.read_csv("experiments/gpt_baselines/results/context_appended/gpt_predictions_with_context.csv")

deberta_df = deberta_df.rename(columns={"prediction_label": "deberta_label"})
gpt_df = gpt_df.rename(columns={"prediction_label": "gpt_label"})

df = df.merge(
    deberta_df[["id", "situation", "deberta_label"]],
    on="id",
    how="left"
)

df = df.merge(
    gpt_df[["id", "gpt_label"]],
    on="id",
    how="left"
)

df.to_csv("data/processed/CS263_dataset_with_predictions.csv", index=False)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# ===== 1. Load CSV =====
input_file = "data/processed/CS263_dataset_with_predictions.csv"
df = pd.read_csv(input_file)

# ===== 2. Extract country from culture column =====
# Example: "US, adult independence norm" -> "US"
df["country"] = df["culture"].str.split(",").str[0].str.strip()

# ===== 3. Normalize labels =====
label_cols = ["label", "deberta_label", "gpt_label"]

for col in label_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

# ===== 4. Accuracy grouped by country =====
summary_rows = []

for country, group in df.groupby("country"):
    deberta_acc = accuracy_score(group["label"], group["deberta_label"])
    gpt_acc = accuracy_score(group["label"], group["gpt_label"])

    summary_rows.append({
        "country": country,
        "n_samples": len(group),
        "deberta_accuracy": deberta_acc,
        "gpt_accuracy": gpt_acc
    })

summary_df = pd.DataFrame(summary_rows).sort_values("country")

print("\n=== Accuracy by Country ===")
print(summary_df)

# ===== 5. Overall accuracy =====
overall = pd.DataFrame([
    {
        "model": "DeBERTa",
        "accuracy": accuracy_score(df["label"], df["deberta_label"])
    },
    {
        "model": "GPT",
        "accuracy": accuracy_score(df["label"], df["gpt_label"])
    }
])

print("\n=== Overall Accuracy ===")
print(overall)

# ===== 6. Full classification report by country =====
for country, group in df.groupby("country"):
    print(f"\n\n================ {country} ================")

    print("\n--- DeBERTa Classification Report ---")
    print(classification_report(
        group["label"],
        group["deberta_label"],
        zero_division=0
    ))

    print("\n--- GPT Classification Report ---")
    print(classification_report(
        group["label"],
        group["gpt_label"],
        zero_division=0
    ))

# ===== 7. Save country-level summary =====
summary_df.to_csv("accuracy_by_country_with_context.csv", index=False)
overall.to_csv("overall_accuracy_with_context.csv", index=False)

print("\nSaved:")
print("- accuracy_by_country_with_context.csv")
print("- overall_accuracy_with_context.csv")